# Exploratory Data Analysis — NYC Airbnb Short-Term Rental Prices

This notebook performs the essential exploratory data analysis for the price estimation pipeline.
We fetch the raw `sample.csv` artifact from Weights & Biases, profile it, identify data quality
issues, and test the basic fixes that will later be re-implemented in the `basic_cleaning`
pipeline step.

**Note:** the download step must have been run first, so that the `sample.csv` artifact exists in W&B:
```bash
mlflow run . -P steps=download
```

## 1. Fetch the data from W&B

We start a W&B run with `save_code=True` so this notebook is uploaded and versioned in W&B,
then download the latest version of the raw data artifact and load it into pandas.

In [ ]:
import wandb
import pandas as pd

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)
df.head()

## 2. Profile the data

We use `ydata-profiling` to generate an automated EDA report directly in the notebook.

Reviewing the report, we identify the following issues:

- **Missing values** in several columns (`name`, `host_name`, `last_review`, `reviews_per_month`).
- The column **`last_review` is stored as a string** instead of a datetime.
- **Outliers in the `price` column**, including listings at $0 and extreme high prices.

After discussing with the stakeholders, we agreed to consider only listings with a nightly
price **between \$10 and \$350**.

In [ ]:
import ydata_profiling

profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

## 3. Apply basic fixes

We apply two fixes:

1. **Drop price outliers** outside the agreed \$10–\$350 range.
2. **Convert `last_review` to datetime.**

**Note:** we deliberately do *not* impute missing values here. Missing value handling will be
performed in the inference pipeline, so the deployed model can handle missing values at
prediction time as well.

In [ ]:
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

## 4. Verify the fixes

We confirm that prices are now within the defined range and that `last_review` is a datetime.

In [ ]:
df.info()

In [ ]:
print(f"Price range: {df['price'].min()} - {df['price'].max()}")
print(f"last_review dtype: {df['last_review'].dtype}")
print(f"Rows remaining: {df.shape[0]}")

## 5. Conclusions

- The raw data contains price outliers and a mistyped date column; both are fixed by the
  cleaning logic above, which will be moved into the `basic_cleaning` pipeline step.
- Missing values remain by design and will be imputed inside the inference pipeline.

We finish the run so all logs and the notebook code are saved to W&B.

In [ ]:
run.finish()